# Pha R (mo rong) — Buoc 3: Danh gia 4 estimator tren du lieu periodic coupling

Chay ca 4 estimator (KSG, Binning, Symbolic can chay LAI vi day la DU LIEU KHAC voi ban VAR - khong the tai su dung ket qua cu) + Amortized (moi train) tren tap test.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os
os.environ.setdefault('JAVA_HOME', r'C:\Program Files\Java\jdk-22')
import numpy as np, pandas as pd
from pathlib import Path

from pqrst.data.synthetic.corpus import load_corpus
from pqrst.estimators.mine.amortized import AmortizedTEEstimator
from pqrst.baselines.ksg import KSGTEEstimator
from pqrst.baselines.binning import BinningTEEstimator
from pqrst.baselines.symbolic_te import SymbolicTEEstimator
from pqrst.evaluation.grid import evaluate_estimators_on_grid, summarize_grid

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
test_windows = load_corpus(str(BASE/'data'/'interim'/'synthetic_corpus_periodic'/'test.npz'))
est_periodic = AmortizedTEEstimator.load(str(BASE/'results'/'checkpoints'/'phi_amortized_periodic.pt'))
print('n_test_windows:', len(test_windows))

n_test_windows: 7200


## 1. Chay danh gia tren toan luoi

In [2]:
estimators = {
    'Amortized': est_periodic,
    'KSG': KSGTEEstimator(),
    'Symbolic': SymbolicTEEstimator(),
    'Binning': BinningTEEstimator(),
}
raw = evaluate_estimators_on_grid(estimators, test_windows, progress=True)
out_dir = BASE / 'results' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)
raw.to_csv(out_dir / 'phase_r2_periodic_grid_raw.csv', index=False)

Evaluating windows: 100%|██████████| 7200/7200 [02:50<00:00, 42.20it/s]


## 2. Ty le loi theo estimator x N

In [3]:
pivot = raw[raw['te_estimate'].isna()].groupby(['estimator','n_samples']).size().unstack(fill_value=0)
display(pivot)

n_samples
estimator


## 3. Luu bang tong hop

In [4]:
summary = summarize_grid(raw)
summary.to_csv(BASE/'results'/'tables'/'phase_r2_periodic_grid_summary.csv', index=False)
summary.head()

,config_name,coupling_c,noise_std,n_samples,estimator,n_valid,n_failed,bias,variance,mse,ci_low,ci_high
0,periodic_k_0.0_noise_0.1,0.0,0.1,10,Amortized,150,0,-0.078754,0.019004,0.025080,-0.100816,-0.056693
1,periodic_k_0.0_noise_0.1,0.0,0.1,10,Binning,150,0,0.263886,0.027525,0.096977,0.237335,0.290437
2,periodic_k_0.0_noise_0.1,0.0,0.1,10,KSG,150,0,0.051423,0.005998,0.008602,0.039029,0.063817
3,periodic_k_0.0_noise_0.1,0.0,0.1,10,Symbolic,150,0,0.052117,0.010843,0.013487,0.035452,0.068781
4,periodic_k_0.0_noise_0.1,0.0,0.1,20,Amortized,150,0,-0.092629,0.013660,0.022149,-0.111333,-0.073925
